# Subtensor Division experiment

This is a simple experiment to see the impact of dividing the latent representation in to subtensors on compression efficiency.

## Load Model

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import torch

from compressai.zoo import bmshj2018_factorized
device = "cuda" if torch.cuda.is_available() else "cpu"
net = bmshj2018_factorized(quality=5, pretrained=True).eval().to(device)

## Load Image

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("images/airplane.bmp").convert("RGB")
x = transforms.ToTensor()(img).unsqueeze(0).to(device)
plt.figure()
plt.axis("off")
plt.imshow(img)
plt.show()

## Inference

In [ ]:
with torch.no_grad():
    y = net.g_a(x)
packet = net.entropy_bottleneck.compress(y)[0]
print(len(packet))
print(y.size())

### Random Split

#### 1. Split along the spatial dimensions

Split the tensor along dim[2].

In [ ]:
import random
height_indices = list(range(32))
random.shuffle(height_indices)

chunk_size = 2
height_chunks = [height_indices[i:i + chunk_size] for i in range(0, len(height_indices), chunk_size)]

print(height_chunks)

In [ ]:
subtensors = []
for chunk in height_chunks:
    subtensor = y[:, :, chunk, :]
    subtensors.append(subtensor)
print(subtensors[0].size())

In [ ]:
packets = []
for subtensor in subtensors:
    packet = net.entropy_bottleneck.compress(subtensor)[0]
    packets.append(packet)
packet_bytes = [len(packet) for packet in packets]
print(packet_bytes)
print(sum(packet_bytes))